# 01 — C. elegans Connectome Exploration

Exploratory analysis of the loaded connectome graph.
This notebook is optional — run `make simulate` for the full emulation.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import torch

from celegans.connectome import load_connectome, build_mock_connectome
from celegans.neuron_types import NeuronType, get_neurons_by_type
from celegans.visualization import plot_connectome_graph, plot_spike_raster

print('Imports OK')

In [ ]:

graph = build_mock_connectome(n_nodes=30, n_edges=120, seed=42)
print(graph.summary())

In [ ]:

for ntype in NeuronType:
    neurons = get_neurons_by_type(ntype)
    print(f'{ntype.value:15s}: {len(neurons):3d} neurons')

In [ ]:

fig = plot_connectome_graph(graph)
plt.show()

In [ ]:

ei = graph.data.edge_index.numpy()
degrees = np.bincount(ei[0], minlength=graph.data.num_nodes)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(degrees, bins=20, color='#4fc3f7', edgecolor='white')
ax.set_xlabel('Out-degree')
ax.set_ylabel('Count')
ax.set_title('Neuron Out-Degree Distribution')
plt.tight_layout()
plt.show()

In [ ]:

from celegans.gnn_model import ConnectomeGNN

model = ConnectomeGNN(
    input_dim=graph.data.x.shape[1],
    hidden_dim=32,
    num_layers=2,
    sensory_indices=graph.get_sensory_indices(),
    motor_indices=graph.get_motor_indices(),
)

with torch.no_grad():
    out = model(graph.data)

print(f'GNN output shape: {out.shape}')
print(f'Output range: [{out.min():.4f}, {out.max():.4f}]')